# inps-pensioni — notebook v0

Notebook v0 di validazione del mart in `dataset-incubator`.

- scopo: sanity check e lettura base del mart
- non sostituisce l'analisi pubblica
- focus iniziale: pensioni sotto i 500 euro, distribuzione territoriale e differenze di genere


In [1]:
import duckdb
from pathlib import Path

MART_TABLE = "mart_pensioni_regione_importo"
METRICA = "numero_pensioni"

candidate_dir = Path.cwd()
if not (candidate_dir / "dataset.yml").exists():
    if (candidate_dir.parent / "dataset.yml").exists():
        candidate_dir = candidate_dir.parent
    else:
        raise FileNotFoundError("dataset.yml non trovato nella directory corrente o nella parent.")

mart_root = candidate_dir.parents[1] / "out" / "data" / "mart" / "inps_pensioni_2017_2024"
mart_candidates = sorted(mart_root.glob(f"*/{MART_TABLE}.parquet"))
if not mart_candidates:
    raise FileNotFoundError("Mart non trovato sotto out/data/mart/inps_pensioni_2017_2024. Eseguire prima toolkit run all.")

PARQUET_PATH = mart_candidates[-1]
relative_parquet = PARQUET_PATH.relative_to(candidate_dir.parents[1])
print(f"Candidate dir: {candidate_dir.name}")
print(f"Parquet path: {relative_parquet}")


Candidate dir: inps-pensioni
Parquet path: out\data\mart\inps_pensioni_2017_2024\2024\mart_pensioni_regione_importo.parquet


In [2]:
con = duckdb.connect()
df = con.execute("SELECT * FROM read_parquet(?)", [str(PARQUET_PATH)]).df()
print(f"Shape: {df.shape}")
display(df.head(10))


Shape: (16453, 8)


,anno,trimestre,area_geografica,regione,sesso,classe_eta,classe_importo,numero_pensioni
0,2020,1,Sud e Isole,Abruzzo,Femmine,55 - 59,"1.000,00 - 1.499,99",38.0
1,2020,1,Sud e Isole,Abruzzo,Femmine,55 - 59,"1.500,00 - 1.999,99",15.0
2,2020,1,Sud e Isole,Abruzzo,Femmine,55 - 59,"2.000,00 - 2.999,99",9.0
3,2020,1,Sud e Isole,Abruzzo,Femmine,55 - 59,"500,00 - 999,99",79.0
4,2020,1,Sud e Isole,Abruzzo,Femmine,55 - 59,"Fino a 499,99",12.0
5,2020,1,Sud e Isole,Abruzzo,Femmine,60 - 64,"1.000,00 - 1.499,99",77.0
6,2020,1,Sud e Isole,Abruzzo,Femmine,60 - 64,"1.500,00 - 1.999,99",90.0
7,2020,1,Sud e Isole,Abruzzo,Femmine,60 - 64,"2.000,00 - 2.999,99",165.0
8,2020,1,Sud e Isole,Abruzzo,Femmine,60 - 64,"500,00 - 999,99",143.0
9,2020,1,Sud e Isole,Abruzzo,Femmine,60 - 64,"Fino a 499,99",22.0


In [3]:
display(df.dtypes)
display(df.isnull().sum())
print(df[METRICA].describe())


anno                 int32
trimestre            int32
area_geografica        str
regione                str
sesso                  str
classe_eta             str
classe_importo         str
numero_pensioni    float64
dtype: object

anno               0
trimestre          0
area_geografica    0
regione            0
sesso              0
classe_eta         0
classe_importo     0
numero_pensioni    0
dtype: int64

count    16453.000000
mean       198.661034
std        340.648889
min          1.000000
25%         14.000000
50%         62.000000
75%        228.000000
max       6343.000000
Name: numero_pensioni, dtype: float64


In [4]:
quota_basse = (
    df[df["classe_importo"] == "Fino a 499,99"]
    .groupby(["anno", "regione"], as_index=False)[METRICA]
    .sum()
)
display(quota_basse.sort_values(["anno", METRICA], ascending=[True, False]).head(20))

gap_genere = (
    df[df["classe_importo"] == "Fino a 499,99"]
    .groupby(["anno", "sesso"], as_index=False)[METRICA]
    .sum()
)
display(gap_genere.sort_values(["anno", "sesso"]))


,anno,regione,numero_pensioni
8,2020,Lombardia,26542.0
3,2020,Campania,18207.0
14,2020,Sicilia,16238.0
6,2020,Lazio,15837.0
12,2020,Puglia,14168.0
4,2020,Emilia Romagna,12919.0
18,2020,Veneto,11616.0
11,2020,Piemonte e Valle dAosta,11473.0
15,2020,Toscana,10781.0
2,2020,Calabria,8415.0


,anno,sesso,numero_pensioni
0,2020,Femmine,105385.0
1,2020,Maschi,78041.0
2,2021,Femmine,109603.0
3,2021,Maschi,79060.0
4,2022,Femmine,109839.0
5,2022,Maschi,80799.0
6,2023,Femmine,87388.0
7,2023,Maschi,61294.0
8,2024,Femmine,19157.0
9,2024,Maschi,13407.0


## Note v0

- Candidate: `inps-pensioni`
- Dataset tecnico: `inps_pensioni_2017_2024`
- Tabella mart: `mart_pensioni_regione_importo`
- Metrica guida: `numero_pensioni`
- Perimetro: regione, sesso, classe età, classe importo, trimestre
- Verifica chiave da fare in lettura: stabilità semantica delle classi di importo lungo la serie 2017-2024
